In [22]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


BD_CRIME = {
    "host": "localhost",
    "port": 5432,
    "database": "crime_spb",
    "user": "postgres",
    "password": "0000",
}


def get_engine():
    url = URL.create(
        drivername="postgresql+psycopg2",
        username=BD_CRIME["user"],
        password=BD_CRIME["password"],
        host=BD_CRIME["host"],
        port=BD_CRIME["port"],
        database=BD_CRIME["database"],
    )
    return create_engine(url)


engine = get_engine()

query = """
    SELECT
        year,
        kpi_code,
        value
    FROM kpi_crime_spb_2ds
    WHERE year IS NOT NULL
      AND kpi_code IS NOT NULL
      AND value IS NOT NULL
    ORDER BY year, kpi_code
"""

crime = pd.read_sql(query, con=engine)

print(crime.head())
print(crime.shape)
print(sorted(crime["year"].unique()))
print(sorted(crime["kpi_code"].unique()))

   year              kpi_code    value
0  2014    victims_rate_total    642.0
1  2014  victims_serious_rate     20.5
2  2014         victims_total  32948.0
3  2015    victims_rate_total    592.0
4  2015  victims_serious_rate     28.6
(41, 3)
[np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
['rape_attempts_cases', 'victims_rate_total', 'victims_serious_rate', 'victims_total']


In [23]:
year_wide = (
    crime.pivot(index="year", columns="kpi_code", values="value")
    .reset_index()
    .sort_values("year")
    .reset_index(drop=True)
)

year_wide.columns.name = None

print(year_wide)
print(year_wide.isna().sum())

    year  rape_attempts_cases  victims_rate_total  victims_serious_rate  \
0   2014                  NaN               642.0                  20.5   
1   2015                  NaN               592.0                  28.6   
2   2016                 69.0               559.0                  25.9   
3   2017                 51.0               571.0                  25.5   
4   2018                 68.0               556.0                  25.7   
5   2019                 60.0               539.0                  22.5   
6   2020                 42.0               747.0                  22.4   
7   2021                 68.0               748.0                  22.6   
8   2022                 56.0               746.0                  19.4   
9   2023                 51.0               754.0                  18.6   
10  2024                 74.0               782.0                  19.4   

    victims_total  
0         32948.0  
1         30717.0  
2         29210.0  
3         30163.0  

In [24]:
year_wide_fill = year_wide.copy()

fill_cols = [
    "rape_attempts_cases",
    "victims_rate_total",
    "victims_serious_rate",
    "victims_total",
]

for col in fill_cols:
    year_wide_fill[col] = year_wide_fill[col].fillna(year_wide_fill[col].mean())

print(year_wide_fill)
print(year_wide_fill.isna().sum())

    year  rape_attempts_cases  victims_rate_total  victims_serious_rate  \
0   2014            59.888889               642.0                  20.5   
1   2015            59.888889               592.0                  28.6   
2   2016            69.000000               559.0                  25.9   
3   2017            51.000000               571.0                  25.5   
4   2018            68.000000               556.0                  25.7   
5   2019            60.000000               539.0                  22.5   
6   2020            42.000000               747.0                  22.4   
7   2021            68.000000               748.0                  22.6   
8   2022            56.000000               746.0                  19.4   
9   2023            51.000000               754.0                  18.6   
10  2024            74.000000               782.0                  19.4   

    victims_total  
0         32948.0  
1         30717.0  
2         29210.0  
3         30163.0  

In [26]:
X = year_wide_fill.drop(columns=["year"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scores = []

for k in [2, 3]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    scores.append((k, score))

scores_df = pd.DataFrame(scores, columns=["k", "silhouette"])
print(scores_df)

   k  silhouette
0  2    0.430416
1  3    0.434896


In [28]:
best_k = int(scores_df.sort_values("silhouette", ascending=False).iloc[0]["k"])

model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
year_wide_fill["cluster"] = model.fit_predict(X_scaled)

print(year_wide_fill[["year", "cluster"]])
print(year_wide_fill.columns)

    year  cluster
0   2014        1
1   2015        1
2   2016        1
3   2017        1
4   2018        1
5   2019        1
6   2020        0
7   2021        2
8   2022        0
9   2023        0
10  2024        2
Index(['year', 'rape_attempts_cases', 'victims_rate_total',
       'victims_serious_rate', 'victims_total', 'cluster'],
      dtype='object')


In [29]:
cluster = (
    year_wide_fill.groupby("cluster")[["rape_attempts_cases",
                                       "victims_rate_total",
                                       "victims_serious_rate",
                                       "victims_total"]]
    .mean()
    .round(1)
)

print(cluster)

         rape_attempts_cases  victims_rate_total  victims_serious_rate  \
cluster                                                                  
0                       49.7               749.0                  20.1   
1                       61.3               576.5                  24.8   
2                       71.0               765.0                  21.0   

         victims_total  
cluster                 
0              40899.7  
1              31462.8  
2              42020.0  


In [30]:
problem_crime_spb2 = year_wide_fill[[
    "year",
    "rape_attempts_cases",
    "victims_rate_total",
    "victims_serious_rate",
    "victims_total",
    "cluster"
]].copy()

print(problem_crime_spb2)

    year  rape_attempts_cases  victims_rate_total  victims_serious_rate  \
0   2014            59.888889               642.0                  20.5   
1   2015            59.888889               592.0                  28.6   
2   2016            69.000000               559.0                  25.9   
3   2017            51.000000               571.0                  25.5   
4   2018            68.000000               556.0                  25.7   
5   2019            60.000000               539.0                  22.5   
6   2020            42.000000               747.0                  22.4   
7   2021            68.000000               748.0                  22.6   
8   2022            56.000000               746.0                  19.4   
9   2023            51.000000               754.0                  18.6   
10  2024            74.000000               782.0                  19.4   

    victims_total  cluster  
0         32948.0        1  
1         30717.0        1  
2         29

In [31]:
problem_crime_spb2.to_sql(
    "problem_crime_spb2",
    con=engine,
    if_exists="replace",
    index=False
)

print("Таблица problem_crime_spb2 сохранена в БД")

Таблица problem_crime_spb2 сохранена в БД


In [32]:
check_df = pd.read_sql(
    "SELECT * FROM problem_crime_spb2 ORDER BY year",
    con=engine
)

print(check_df)

    year  rape_attempts_cases  victims_rate_total  victims_serious_rate  \
0   2014            59.888889               642.0                  20.5   
1   2015            59.888889               592.0                  28.6   
2   2016            69.000000               559.0                  25.9   
3   2017            51.000000               571.0                  25.5   
4   2018            68.000000               556.0                  25.7   
5   2019            60.000000               539.0                  22.5   
6   2020            42.000000               747.0                  22.4   
7   2021            68.000000               748.0                  22.6   
8   2022            56.000000               746.0                  19.4   
9   2023            51.000000               754.0                  18.6   
10  2024            74.000000               782.0                  19.4   

    victims_total  cluster  
0         32948.0        1  
1         30717.0        1  
2         29

In [34]:
problem_crime_spb2 = problem_crime_spb2.round(1)
print(problem_crime_spb2)

    year  rape_attempts_cases  victims_rate_total  victims_serious_rate  \
0   2014                 59.9               642.0                  20.5   
1   2015                 59.9               592.0                  28.6   
2   2016                 69.0               559.0                  25.9   
3   2017                 51.0               571.0                  25.5   
4   2018                 68.0               556.0                  25.7   
5   2019                 60.0               539.0                  22.5   
6   2020                 42.0               747.0                  22.4   
7   2021                 68.0               748.0                  22.6   
8   2022                 56.0               746.0                  19.4   
9   2023                 51.0               754.0                  18.6   
10  2024                 74.0               782.0                  19.4   

    victims_total  cluster  
0         32948.0        1  
1         30717.0        1  
2         29

In [35]:
problem_crime_spb2.to_sql(
    "problem_crime_spb2",
    con=engine,
    if_exists="replace",
    index=False
)

print("Таблица problem_crime_spb2 обновлена в БД")

Таблица problem_crime_spb2 обновлена в БД
